# Phase 3: registered sample-size sweep

Sixty gate runs first: cells A to D at 50 images, five folds, seeds 42, 43, 44.
The stability gate pools the inner-validation DKA-effect variance; the sweep
continues when the pooled standard deviation is under 0.030. A passed gate adds
eighty runs (100 images with seeds 42 and 43, then 251 and 501 with seed 42).
C4 is the slope of the DKA effect on log10(n), with the 1,044-image effects
read from Phase 2. A stopped gate reports C4 as not estimated and keeps the
four-test Bonferroni correction.


In [1]:
%run definitions.ipynb
require_pythonhashseed()
PH2 = PROJECT_ROOT / "phase2" / "outputs"
PH3 = PROJECT_ROOT / "phase3" / "outputs"
PH3.mkdir(parents=True, exist_ok=True)
ENV = environment_record()
SPLITS_CTX = load_split_tables()
COHORT, FOLDS, INNER, LABELS, GROUPS = SPLITS_CTX
p2_manifest = manifest_frame(PH2)
P2_DONE = (p2_manifest["status"] == "success").sum() >= 35 if len(p2_manifest) else False
if not P2_DONE:
    print("NOTE: Phase 2 is not complete. Phase 3 starts after it.")


required packages already match the pins.
torch 2.8.0+cu128 present (training phases need >= 2.8.0).
torchvision 0.23.0+cu128 present (training phases need >= 0.23.0).


python 3.12.3 | numpy 2.2.6 | pandas 2.3.3 | pillow 12.0.0 | opencv 4.12.0 | torch available: True


In [2]:
GATE_STATUS = "blocked"
if P2_DONE:
    GATE_STATUS = execute_pending_runs(PH3, phase3_gate_specs(), SPLITS_CTX, "phase3.ipynb")
else:
    print("PENDING: waiting for Phase 2.")


[1/60] p3_A_f1_s42_n50


[2/60] p3_A_f1_s43_n50


[3/60] p3_A_f1_s44_n50


[4/60] p3_A_f2_s42_n50


[5/60] p3_A_f2_s43_n50


...

[56/60] p3_D_f4_s43_n50


[57/60] p3_D_f4_s44_n50


[58/60] p3_D_f5_s42_n50


[59/60] p3_D_f5_s43_n50


[60/60] p3_D_f5_s44_n50


In [3]:
# Stability gate on inner-validation DKA effects at 50 images.
GATE_PASSED = None
if GATE_STATUS == "complete":
    inner_auc = {}
    for spec in phase3_gate_specs():
        meta = json.load(open(PH3 / "runs" / spec["run_id"] / "meta.json"))
        inner_auc[(spec["cell"], spec["fold"], spec["seed"])] = meta["best_inner_macro_auc"]
    per_fold_sd = []
    fold_effects = {}
    for f in range(1, N_FOLDS + 1):
        effects = [0.5 * ((inner_auc[("B", f, s)] - inner_auc[("A", f, s)])
                          + (inner_auc[("D", f, s)] - inner_auc[("C", f, s)]))
                   for s in [42, 43, 44]]
        fold_effects[f] = effects
        per_fold_sd.append(float(np.std(effects, ddof=1)))
    sd50 = pooled_sd(per_fold_sd, seeds_minus_one=2, dof=10)
    GATE_PASSED = bool(sd50 < 0.030)
    gate = {"sd50": sd50, "threshold": 0.030, "passed": GATE_PASSED,
            "per_fold_sd": per_fold_sd,
            "fold_effects": {str(k): v for k, v in fold_effects.items()}}
    write_json(PH3 / "gate_result.json", gate)
    verdict = "continue the sweep" if GATE_PASSED else "stop the sweep, report instability"
    print(f"SD(50) = {sd50:.4f} against 0.030: {verdict}")
else:
    print("PENDING: the gate waits for the 60 gate runs.")


SD(50) = 0.0036 against 0.030: continue the sweep


In [4]:
CONT_STATUS = "blocked"
if GATE_PASSED:
    CONT_STATUS = execute_pending_runs(PH3, phase3_continuation_specs(), SPLITS_CTX,
                                       "phase3.ipynb")
elif GATE_PASSED is None:
    print("PENDING: continuation waits for the gate.")
else:
    CONT_STATUS = "stopped"
    print("gate stopped the sweep: the 80 continuation runs do not execute.")


[1/80] p3_A_f1_s42_n100


[2/80] p3_A_f1_s43_n100


[3/80] p3_A_f1_s42_n251


[4/80] p3_A_f1_s42_n501


[5/80] p3_A_f2_s42_n100


...

[76/80] p3_D_f4_s42_n501


[77/80] p3_D_f5_s42_n100


[78/80] p3_D_f5_s43_n100


[79/80] p3_D_f5_s42_n251


[80/80] p3_D_f5_s42_n501


In [5]:
# C4 preparation: load every ladder run, define the effect pipeline, compute
# the point estimate, and verify serial-versus-parallel parity on 500
# replicates. LADDER mirrors the registered seed schedule.
LADDER = {50: [42, 43, 44], 100: [42, 43], 251: [42], 501: [42], 1044: [42]}

def _run_dir_for(n, cell, fold, seed):
    if n == 1044:
        return PH2, f"p2_{cell}_f{fold}_s{seed}_n1044"
    return PH3, f"p3_{cell}_f{fold}_s{seed}_n{n}"

C4_READY = False
if GATE_PASSED and CONT_STATUS == "complete":
    preds = {}
    for n, seeds in LADDER.items():
        for cell in ["A", "B", "C", "D"]:
            for f in range(1, N_FOLDS + 1):
                for s in seeds:
                    base, rid = _run_dir_for(n, cell, f, s)
                    preds[(n, cell, f, s)] = load_test_predictions(base, rid)

    fold_ids = {f: preds[(1044, "A", f, 42)]["image_ids"] for f in range(1, N_FOLDS + 1)}
    fold_labels = {f: preds[(1044, "A", f, 42)]["labels"] for f in range(1, N_FOLDS + 1)}
    fold_groups = {f: fold_group_index_lists(fold_ids[f], GROUPS) for f in fold_ids}

    def effects_by_n(index_of_fold=None, masks_of_fold=None):
        out = {}
        for n, seeds in LADDER.items():
            fold_means = []
            for f in range(1, N_FOLDS + 1):
                idx = None if index_of_fold is None else index_of_fold[f]
                included = None if masks_of_fold is None else                     [k for k in range(len(TARGETS)) if masks_of_fold[f][k]]
                seed_effects = []
                for s in seeds:
                    auc = {}
                    for cell in ["A", "B", "C", "D"]:
                        p = preds[(n, cell, f, s)]
                        if idx is None:
                            auc[cell] = macro_auc(p["labels"], p["probs"])[0]
                        else:
                            auc[cell] = macro_auc(p["labels"][idx], p["probs"][idx], included)[0]
                    seed_effects.append(dka_effect(auc))
                fold_means.append(float(np.mean(seed_effects)))
            out[n] = float(np.mean(fold_means))
        return out

    def make_c4_replicate(reps_local, masks_local):
        def _replicate(r):
            return c4_slope(effects_by_n(reps_local[r], masks_local[r]))
        return _replicate

    point_effects = effects_by_n()
    c4_hat = c4_slope(point_effects)

    reps500, masks500, _ = draw_replicates(fold_groups, 500, BOOTSTRAP_SEED, fold_labels)
    check_fn = make_c4_replicate(reps500, masks500)
    t0 = time.time()
    serial_out = run_replicates(check_fn, 500, parallel=False)
    t1 = time.time()
    forked_out = run_replicates(check_fn, 500)
    t2 = time.time()
    if serial_out != forked_out:
        raise PhaseStop("parallel bootstrap output differs from the serial output.")
    print(f"parity check passed: 500 replicates bit-identical "
          f"(serial {t1 - t0:.1f}s, parallel {t2 - t1:.1f}s, {os.cpu_count()} cpus)")
    C4_READY = True
else:
    print("PENDING: the C4 preparation waits for the sweep runs.")


parity check passed: 500 replicates bit-identical (serial 31.5s, parallel 29.7s, 255 cpus)


In [6]:
# C4 full bootstrap in parallel, plus the SD(100)/SD(50) diagnostic.
if C4_READY:
    print("drawing bootstrap replicates...")
    reps, masks, discarded = draw_replicates(fold_groups, BOOTSTRAP_REPLICATES,
                                             BOOTSTRAP_SEED, fold_labels)
    rep_fn = make_c4_replicate(reps, masks)
    t0 = time.time()
    c4_boot = run_replicates(rep_fn, len(reps))
    print(f"{len(c4_boot)} replicates computed in {time.time() - t0:.0f}s")
    c4_summary = bootstrap_summary("C4", c4_hat, c4_boot)

    inner_auc100 = {}
    for cell in ["A", "B", "C", "D"]:
        for f in range(1, N_FOLDS + 1):
            for s in [42, 43]:
                meta = json.load(open(PH3 / "runs" / f"p3_{cell}_f{f}_s{s}_n100" / "meta.json"))
                inner_auc100[(cell, f, s)] = meta["best_inner_macro_auc"]
    sd100_folds = []
    for f in range(1, N_FOLDS + 1):
        eff = [0.5 * ((inner_auc100[("B", f, s)] - inner_auc100[("A", f, s)])
                      + (inner_auc100[("D", f, s)] - inner_auc100[("C", f, s)]))
               for s in [42, 43]]
        sd100_folds.append(float(np.std(eff, ddof=1)))
    sd100 = pooled_sd(sd100_folds, seeds_minus_one=1, dof=5)
    gate = json.load(open(PH3 / "gate_result.json"))
    result = {"estimated": True, "C4": c4_summary,
              "effects_by_n": {str(k): v for k, v in point_effects.items()},
              "discarded_replicates": discarded,
              "sd100": sd100, "sd50": gate["sd50"],
              "sd_ratio_100_over_50": sd100 / gate["sd50"]}
    write_json(PH3 / "analysis_c4.json", result)
    print(json.dumps({k: result[k] for k in ["C4", "effects_by_n", "sd_ratio_100_over_50"]},
                     indent=2))
elif GATE_PASSED is False:
    write_json(PH3 / "analysis_c4.json",
               {"estimated": False,
                "reason": "the stability gate stopped the sweep",
                "note": "C1-C3 keep the four-test Bonferroni correction"})
    print("C4 recorded as not estimated.")
else:
    print("PENDING: the C4 analysis waits for the sweep runs.")


drawing bootstrap replicates...


10000 replicates computed in 66s
{
  "C4": {
    "contrast": "C4",
    "estimate": -0.0062768413271971,
    "ci_low": -0.010619164346600624,
    "ci_high": -0.0019719802399320347,
    "p_raw": 0.0008999100089991,
    "p_adjusted": 0.0035996400359964,
    "replicates": 10000
  },
  "effects_by_n": {
    "50": -0.0014279378453987634,
    "100": 0.0003163068693685822,
    "251": 0.00014068186440898688,
    "501": -0.0031616193455933027,
    "1044": -0.010361395958147612
  },
  "sd_ratio_100_over_50": 1.0460927549890262
}


In [7]:
manifest = manifest_frame(PH3)
ok = (manifest["status"] == "success").sum() if len(manifest) else 0
if (PH3 / "analysis_c4.json").exists():
    c4 = json.load(open(PH3 / "analysis_c4.json"))
    if c4["estimated"] and ok >= 140:
        print("140 successful runs and a complete C4 result. PHASE 3 COMPLETE")
    elif not c4["estimated"] and ok >= 60:
        print("60 successful runs and a not-estimated C4 record. PHASE 3 COMPLETE")
    else:
        print(f"PHASE 3 PENDING: {ok} successful runs so far.")
else:
    print(f"PHASE 3 PENDING: {ok} successful runs so far.")


140 successful runs and a complete C4 result. PHASE 3 COMPLETE
